# Module 5 — ML Forecasting with LightGBM
**Type:** [Code With Me]  
**Time:** 45 minutes  
**Job:** Prove that a feature-engineered ML pipeline beats the statistical baselines. Quantify the compute cost. Make an honest recommendation about whether the improvement justifies the infrastructure.

---
## 5.1 — Setup
**[Watch Only]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast

print("Setup complete.")
print(f"USE_TUNED_PARAMS = {USE_TUNED_PARAMS}")

---
## 5.2 — Load Panel and Micro Subset
**[Watch Only]**

In [ ]:
panel = load_checkpoint("03_validated_panel")

top_series = (
    panel.groupby("unique_id")["y"]
    .sum()
    .sort_values(ascending=False)
    .head(MICRO_SUBSET_N)
    .index
)
micro = panel[panel["unique_id"].isin(top_series)].copy()

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 5.3 — Why ML Forecasting?
**[Watch Only]**

AutoETS is a strong univariate model. It sees one series at a time. It cannot learn patterns that span series — shared seasonal profiles, cross-SKU promotional lifts, or price elasticity signals.

**LightGBM via MLForecast is a global model.** It trains on all series simultaneously. Each series becomes a set of training rows. The model learns patterns across the entire category portfolio.

**The compute tradeoff is real:**  
AutoETS on 1,000 series takes seconds. LightGBM on 1,000 series with lag features takes longer and produces a model artifact you now have to version, deploy, and monitor. The accuracy gain must justify that operational cost.

**The features we will build:**

| Feature type | Examples | Business signal |
|---|---|---|
| Autoregressive lags | lag_7, lag_14, lag_28 | Recent demand history |
| Rolling statistics | 28-day rolling mean | Trend smoothing |
| Date features | day_of_week, month | Calendar seasonality |

Adding lags in a notebook is a one-liner. Maintaining a feature store that recomputes these lags daily across 100,000 SKUs — with late data, backfills, and audit trails — is a data engineering project that takes months.

---
## 5.4 — Load LightGBM Parameters
**[Watch Only]**

In [ ]:
params_file = "mlforecast_lgb_tuned.json" if USE_TUNED_PARAMS else "mlforecast_lgb_default.json"
params_path = PARAMS_DIR / params_file

with open(params_path) as f:
    lgb_params = json.load(f)

lgb_params["random_state"] = RANDOM_SEED

print(f"Loading params from: {params_file}")
for k, v in lgb_params.items():
    print(f"  {k:<22}: {v}")

---
## 5.5 — Configure MLForecast
**[Code With Me — 3 lines]**

Fill in `lags`, `lag_transforms`, and `date_features`. Use weekly, biweekly, and monthly lags. Rolling mean on lag 7 with a 28-day window. Day-of-week and month as date features.

In [ ]:
mlf = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=__FILL_IN__,                    # [7, 14, 28]
    lag_transforms={
        7: __FILL_IN__,                  # [RollingMean(window_size=28)]
    },
    date_features=__FILL_IN__,           # ["dayofweek", "month"]
)

print("MLForecast configured.")
print(f"  Lags          : {mlf.ts.lags}")
print(f"  Date features : {mlf.ts.date_features}")
print(f"  Model         : {mlf.models[0].__class__.__name__}")

**Expected output:**
```
MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor
```

---
## 5.6 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

In [ ]:
%%time
# Cross-validation on micro subset
# Target runtime: < 30 seconds on Colab CPU

cv_ml_micro = mlf.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=REFIT,
)

print(f"ML CV complete. Shape: {cv_ml_micro.shape}")
print(f"Columns: {list(cv_ml_micro.columns)}")
print(cv_ml_micro.head(3).to_string())

**Expected output:**
```
ML CV complete. Shape: (4200, 5)
Columns: ['ds', 'cutoff', 'y', 'LGBMRegressor']
```

---
## 5.7 — Reshape and Add Conformal Prediction Intervals
**[Code With Me — 2 lines]**

Fill in the residual calculation (`y - y_hat`) and the 10th percentile extraction. The 90th percentile is already provided — match the pattern.

In [ ]:
def reshape_mlforecast_cv(cv_df: pd.DataFrame, model_col: str, stage: str) -> pd.DataFrame:
    """
    Reshape MLForecast wide CV output to forecast schema.
    Adds conformal prediction intervals derived from in-sample residuals.
    """
    df = cv_df.reset_index().copy()
    df = df.rename(columns={model_col: "y_hat"})
    df["model"] = "LightGBM"
    df["stage"] = stage
    df["y_hat"] = df["y_hat"].clip(lower=0)

    # Conformal prediction intervals from in-sample residuals
    residuals  = __FILL_IN__                         # df["y"] - df["y_hat"]
    lo_offset  = __FILL_IN__                         # np.percentile(residuals, 10)
    hi_offset  = np.percentile(residuals, 90)

    df["lo_80"] = (df["y_hat"] + lo_offset).clip(lower=0)
    df["hi_80"] = (df["y_hat"] + hi_offset).clip(lower=0)

    return df[["unique_id", "ds", "y", "model", "y_hat", "lo_80", "hi_80", "cutoff", "stage"]]


ml_micro = reshape_mlforecast_cv(cv_ml_micro, model_col="LGBMRegressor", stage="ml")

print(f"Reshaped ML forecasts: {ml_micro.shape}")
print(f"Columns: {list(ml_micro.columns)}")
print(f"Interval sample (lo_80, y_hat, hi_80):")
print(ml_micro[["lo_80", "y_hat", "hi_80"]].describe().round(2).to_string())

**Expected output:**
```
Reshaped ML forecasts: (4200, 9)
Columns: ['unique_id', 'ds', 'y', 'model', 'y_hat', 'lo_80', 'hi_80', 'cutoff', 'stage']
```

---
## 5.8 — Plot: LightGBM vs Best Baseline
**[Watch Only]**

In [ ]:
# Compare LightGBM vs SeasonalNaive on a single series
sample_uid = top_series[0]
sample_cut = ml_micro["cutoff"].unique()[-1]

actuals   = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
plot_start = sample_cut - pd.Timedelta(days=42)

lgbm_fcast = ml_micro[
    (ml_micro["unique_id"] == sample_uid) &
    (ml_micro["cutoff"]    == sample_cut)
].set_index("ds")

try:
    baseline_micro_path = ARTIFACT_DIR / "04_baseline_micro_forecasts.parquet"
    baseline_micro = pd.read_parquet(baseline_micro_path)
    snaive_fcast = baseline_micro[
        (baseline_micro["unique_id"] == sample_uid) &
        (baseline_micro["cutoff"]    == sample_cut) &
        (baseline_micro["model"]     == "SeasonalNaive")
    ].set_index("ds")
    has_baseline = True
except Exception:
    has_baseline = False

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actuals[actuals.index >= plot_start].index,
        actuals[actuals.index >= plot_start].values,
        color="#333", linewidth=1.0, label="Actual", zorder=3)
ax.plot(lgbm_fcast.index, lgbm_fcast["y_hat"],
        color="#7B1FA2", linewidth=1.5, linestyle="--", label="LightGBM", zorder=4)
ax.fill_between(lgbm_fcast.index, lgbm_fcast["lo_80"], lgbm_fcast["hi_80"],
                alpha=0.15, color="#7B1FA2", label="LightGBM 80% interval")
if has_baseline and len(snaive_fcast) > 0:
    ax.plot(snaive_fcast.index, snaive_fcast["y_hat"],
            color="#1E88E5", linewidth=1.2, linestyle=":", label="SeasonalNaive", zorder=3)
ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
ax.set_title(f"LightGBM vs SeasonalNaive — {sample_uid} (Window 3)", fontsize=11)
ax.set_ylabel("Units sold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Expected output:** LightGBM forecast with shaded 80% interval alongside SeasonalNaive. Compare interval width — narrower intervals with fewer misses = better uncertainty quantification.

---
## 5.9 — Score the Micro ML Forecasts
**[Code With Me — 1 line]**

Fill in the `validate_forecast` call before scoring.

In [ ]:
# Validate schema before scoring
ml_micro_validated = __FILL_IN__  # validate_forecast(ml_micro, artifact_name="05_ml_micro")

ml_micro_scores = score_forecasts(
    ml_micro_validated,
    subset_name=f"micro_{MICRO_SUBSET_N}",
)

wmape_row  = ml_micro_scores[ml_micro_scores["metric"] == "wMAPE"].iloc[0]
iscore_row = ml_micro_scores[ml_micro_scores["metric"] == "IntervalScore_80"].iloc[0]

print(f"LightGBM on micro subset ({MICRO_SUBSET_N} series):")
print(f"  wMAPE          : {wmape_row['score']:.4f}")
print(f"  Interval Score : {iscore_row['score']:.4f}")

**Expected output:**
```
LightGBM on micro subset (50 series):
  wMAPE          : 0.XXXX
  Interval Score : X.XXXX
```

---
## 5.10 — Load Full-Subset ML Forecasts and Update Leaderboard
**[Watch Only]**

In [ ]:
# 🔴 RED PATH (standard) — load precomputed full-subset ML forecasts
ml_full = load_checkpoint("05_ml_forecasts")

ml_full_scores = score_forecasts(
    ml_full,
    subset_name=f"workshop_{WORKSHOP_SUBSET_N}",
)

baseline_full   = load_checkpoint("04_baseline_forecasts")
baseline_scores = score_forecasts(
    baseline_full,
    subset_name=f"workshop_{WORKSHOP_SUBSET_N}",
)

from src.evaluation import build_leaderboard
leaderboard_5 = build_leaderboard([baseline_scores, ml_full_scores])

print("Updated leaderboard (wMAPE):")
wmape_lb = leaderboard_5[["model", "wMAPE"]].dropna().sort_values("wMAPE") if "wMAPE" in leaderboard_5.columns else leaderboard_5
print(wmape_lb.to_string(index=False))

**Expected output:**
```
Updated leaderboard (wMAPE):
          model    wMAPE
       LightGBM   0.XXXX
        AutoETS   0.XXXX
  SeasonalNaive   0.XXXX
Chronos-t5-mini   0.XXXX
          Naive   0.XXXX
```

---
## 5.11 — Save the Micro ML Artifact
**[Watch Only]**

In [ ]:
micro_ml_path = ARTIFACT_DIR / "05_ml_micro_forecasts.parquet"
ml_micro_validated.to_parquet(micro_ml_path, index=False)
print(f"  ✓ Micro ML artifact saved : {micro_ml_path.name} ({len(ml_micro_validated):,} rows)")
print(f"  ✓ Full ML artifact loaded : 05_ml_forecasts.parquet ({len(ml_full):,} rows)")

---
## 5.12 — The Enterprise Cliff
**[Watch Only]**

Adding lags in a notebook is a one-liner. **Orchestrating a scalable feature pipeline across 100,000 SKUs is where enterprise forecasting systems diverge from standalone scripts.**

What we simplified:

**Lag computation at scale:**  
MLForecast computes lags on the fly during CV. In production, lags are computed once, stored in a feature store, versioned, and served to the model at inference time. Late-arriving data invalidates precomputed lags — your pipeline needs a backfill strategy.

**Conformal intervals are a simplification:**  
The residual-based intervals we computed assume that in-sample errors are representative of out-of-sample errors. That holds on stable series. On series with trend shifts or promotional events, it breaks. Production-grade conformal calibration requires held-out calibration sets, not in-sample residuals.

**The ROI question:**  
LightGBM beat AutoETS by some margin on wMAPE. The question your CFO asks is: how many basis points of safety stock reduction does that margin translate to? If the answer is less than the infrastructure cost of maintaining a feature store, AutoETS is the right production choice.